# GPT2 TTS Training

In [1]:
from dataclasses import dataclass, asdict
from pathlib import Path
import hashlib
import os
import random

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint
from transformers import GPT2Model, GPT2TokenizerFast
from focalcodec import FocalCodec

if torch.cuda.is_available():
    torch.set_float32_matmul_precision("high")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)



cuda


In [2]:
@dataclass
class ExperimentConfig:
    dataset_dir: Path = Path("dataset")
    cache_dir: Path = Path("artifacts/token_cache")
    checkpoint_dir: Path = Path("artifacts/checkpoints")
    seed: int = 42
    val_fraction: float = 0.1
    max_total_tokens: int = 768
    codec_name: str = "lucadellalib/focalcodec_25hz"
    base_model: str = "gpt2"
    encode_batch_size: int = 8
    train_batch_size: int = 8
    accumulate_grad_batches: int = 2
    num_workers: int = 2
    learning_rate: float = 5e-5
    weight_decay: float = 0.01
    max_epochs: int = 10
    gradient_clip_val: float = 1.0
    precision: str = "bf16-mixed" if torch.cuda.is_available() else "32-true"
    freeze_bottom_n_layers: int = 2
    tie_audio_weights: bool = True
    gradient_checkpointing: bool = True
    warmup_fraction: float = 0.1


cfg = ExperimentConfig()
cfg.cache_dir.mkdir(parents=True, exist_ok=True)
cfg.checkpoint_dir.mkdir(parents=True, exist_ok=True)

random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
pl.seed_everything(cfg.seed, workers=True)

Seed set to 42


42

## Data

In [3]:
metadata_path = cfg.dataset_dir / "metadata.csv"
wavs_dir = cfg.dataset_dir / "wavs"

meta = pd.read_csv(metadata_path, sep="|", header=None, names=["id", "text", "text_norm"])
meta["text_norm"] = meta["text_norm"].fillna(meta["text"])
meta["filepath"] = meta["id"].apply(lambda item_id: str(wavs_dir / f"{item_id}.wav"))
meta = meta[meta["filepath"].map(lambda p: Path(p).exists())].reset_index(drop=True)

rng = np.random.default_rng(cfg.seed)
perm = rng.permutation(len(meta))
split_at = int((1.0 - cfg.val_fraction) * len(meta))
train_df = meta.iloc[perm[:split_at]].reset_index(drop=True)
val_df = meta.iloc[perm[split_at:]].reset_index(drop=True)


print(f"Total: {len(meta)}, Train: {len(train_df)}, Val: {len(val_df)}")
train_df.head()

Total: 13100, Train: 11790, Val: 1310


,id,text,text_norm,filepath
0,LJ022-0092,and such projects as they approve will be next...,and such projects as they approve will be next...,dataset/wavs/LJ022-0092.wav
1,LJ039-0028,"was shipped from Los Angeles on March 20, 1963...","was shipped from Los Angeles on March twenty, ...",dataset/wavs/LJ039-0028.wav
2,LJ006-0017,with those who made the selection of the first...,with those who made the selection of the first...,dataset/wavs/LJ006-0017.wav
3,LJ016-0417,"Catherine Wilson, the poisoner, was reserved a...","Catherine Wilson, the poisoner, was reserved a...",dataset/wavs/LJ016-0417.wav
4,LJ012-0101,They had had serious work to get at the diamon...,They had had serious work to get at the diamon...,dataset/wavs/LJ012-0101.wav


## Codec

In [4]:
codec = FocalCodec.from_pretrained(cfg.codec_name).to(DEVICE).eval()
for parameter in codec.parameters():
    parameter.requires_grad_(False)

CODEBOOK_SIZE = int(codec.codebook.shape[0])
CODEC_SR = int(codec.sample_rate_input)
print(f"Codebook Size: {CODEBOOK_SIZE}, Input Sample Rate: {CODEC_SR}, Output Sample Rate: {codec.sample_rate_output}")

Codebook Size: 8192, Input Sample Rate: 16000, Output Sample Rate: 16000


In [5]:
def load_wav_16k(path):
    wav, sample_rate = sf.read(path, dtype="float32", always_2d=False)
    if wav.ndim > 1:
        wav = wav.mean(axis=-1)
    if sample_rate != CODEC_SR:
        wav = librosa.resample(wav, orig_sr=sample_rate, target_sr=CODEC_SR)
    return torch.from_numpy(np.asarray(wav, dtype=np.float32))


example = train_df.iloc[0]
example_wav = load_wav_16k(example["filepath"]).to(DEVICE)
with torch.no_grad():
    example_tokens = codec.sig_to_toks(example_wav.unsqueeze(0)).squeeze(0)

print(example["id"])
print(example["text_norm"])
print(example_wav.numel() / CODEC_SR, example_tokens.numel())

LJ022-0092
and such projects as they approve will be next submitted to the President who under the Act is required to make final allocations.
9.7479375 244


## Token Cache

In [6]:
tokenizer = GPT2TokenizerFast.from_pretrained(cfg.base_model)
tokenizer.pad_token = tokenizer.eos_token

CACHE_DIR = cfg.cache_dir


def split_cache_path(name, df):
    ids = "\n".join(df["id"].astype(str).tolist()).encode("utf-8")
    digest = hashlib.sha1(ids).hexdigest()[:10]
    return CACHE_DIR / f"{name}_{len(df)}_{digest}.pt"


TRAIN_CACHE = split_cache_path("train_items", train_df)
VAL_CACHE = split_cache_path("val_items", val_df)
print(TRAIN_CACHE.name, VAL_CACHE.name)

@torch.no_grad()
def encode_split(df, cache_path, batch_size=16, overwrite=False):
    if cache_path.exists() and not overwrite:
        return torch.load(cache_path, weights_only=False)

    recs = df.to_dict("records")
    paths = [r["filepath"] for r in recs]
    order = np.argsort([os.path.getsize(p) for p in paths])
    out = [None] * len(recs)

    for start in tqdm(range(0, len(recs), batch_size), desc=Path(cache_path).name):
        idxs = order[start:start + batch_size]
        wavs = [load_wav_16k(paths[i]) for i in idxs]
        lens = torch.tensor([w.numel() for w in wavs], dtype=torch.float32)
        L = int(lens.max().item())
        batch = torch.zeros(len(wavs), L, dtype=torch.float32)

        for j, w in enumerate(wavs):
            batch[j, :w.numel()] = w

        length = (lens / L).to(DEVICE)
        toks = codec.sig_to_toks(batch.to(DEVICE), length=length)
        tok_lens = (length * toks.shape[-1]).round().clamp(1, toks.shape[-1]).long()

        for j, i in enumerate(idxs):
            text_ids = tokenizer.encode(recs[i]["text_norm"], add_special_tokens=False)
            text_ids = torch.tensor(text_ids, dtype=torch.long)
            audio_ids = toks[j, :tok_lens[j]].detach().cpu().long()
            out[i] = {
                "id": recs[i]["id"],
                "text": recs[i]["text_norm"],
                "text_ids": text_ids,
                "audio_ids": audio_ids,
                "n_text": int(text_ids.numel()),
                "n_audio": int(audio_ids.numel()),
            }

    out = [item for item in out if item is not None]
    torch.save(out, cache_path)
    return out

train_items_11790_8ee4f5215e.pt val_items_1310_9d1962113b.pt


In [7]:
train_items = encode_split(train_df, TRAIN_CACHE, batch_size=cfg.encode_batch_size)
val_items = encode_split(val_df, VAL_CACHE, batch_size=cfg.encode_batch_size)

print(f"Train Items: {len(train_items)}, Val Items: {len(val_items)}")
print(f"Average Audio Tokens: {np.mean([item['n_audio'] for item in train_items])}")
print(f"Average Text Tokens: {np.mean([item['n_text'] for item in train_items])}")

Train Items: 11790, Val Items: 1310
Average Audio Tokens: 164.69821882951655
Average Text Tokens: 20.808227311280746


## Dataset

In [8]:
class TokenizedLJSpeech(Dataset):
    def __init__(self, cache_path, max_total_len=1024):
        items = torch.load(cache_path, weights_only=False)
        self.items = [it for it in items if it["n_text"] + it["n_audio"] + 2 <= max_total_len]

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        it = self.items[i]
        return {
            "id": it["id"],
            "text": it["text"],
            "text_ids": it["text_ids"].long(),
            "audio_ids": it["audio_ids"].long(),
        }


def collate(batch):
    B = len(batch)
    text_lens = torch.tensor([b["text_ids"].numel() for b in batch], dtype=torch.long)
    audio_lens = torch.tensor([b["audio_ids"].numel() for b in batch], dtype=torch.long)
    text_ids = torch.zeros(B, int(text_lens.max()), dtype=torch.long)
    audio_ids = torch.zeros(B, int(audio_lens.max()), dtype=torch.long)

    for i, b in enumerate(batch):
        text_ids[i, :b["text_ids"].numel()] = b["text_ids"]
        audio_ids[i, :b["audio_ids"].numel()] = b["audio_ids"]

    return {
        "ids": [b["id"] for b in batch],
        "texts": [b["text"] for b in batch],
        "text_ids": text_ids,
        "text_lens": text_lens,
        "audio_ids": audio_ids,
        "audio_lens": audio_lens,
    }

In [9]:
train_ds = TokenizedLJSpeech(TRAIN_CACHE, max_total_len=cfg.max_total_tokens)
val_ds = TokenizedLJSpeech(VAL_CACHE, max_total_len=cfg.max_total_tokens)

train_dl = DataLoader(
    train_ds,
    batch_size=cfg.train_batch_size,
    shuffle=True,
    collate_fn=collate,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
)

val_dl = DataLoader(
    val_ds,
    batch_size=cfg.train_batch_size,
    shuffle=False,
    collate_fn=collate,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
)

batch = next(iter(train_dl))
{k: tuple(v.shape) for k, v in batch.items() if torch.is_tensor(v)}

{'text_ids': (8, 36),
 'text_lens': (8,),
 'audio_ids': (8, 249),
 'audio_lens': (8,)}

## Model

In [10]:
@dataclass
class GPT2TTSConfig:
    base_model: str = "gpt2"
    codebook_size: int = 8192
    n_special_tokens: int = 2
    freeze_bottom_n_layers: int = 0
    tie_audio_weights: bool = True
    gradient_checkpointing: bool = False

    @property
    def audio_vocab_size(self) -> int:
        return self.codebook_size + self.n_special_tokens

    @property
    def bos_id(self) -> int:
        return self.codebook_size

    @property
    def eos_id(self) -> int:
        return self.codebook_size + 1

In [11]:
class GPT2TTS(nn.Module):
    def __init__(self, cfg: GPT2TTSConfig):
        super().__init__()
        self.cfg = cfg
        self.base = GPT2Model.from_pretrained(cfg.base_model)
        H = self.base.config.n_embd
        V = cfg.audio_vocab_size
        self.audio_emb = nn.Embedding(V, H)
        self.audio_head = nn.Linear(H, V, bias=False)
        nn.init.normal_(self.audio_emb.weight, std=0.02)
        if cfg.tie_audio_weights:
            self.audio_head.weight = self.audio_emb.weight
        else:
            nn.init.normal_(self.audio_head.weight, std=0.02)

        if cfg.gradient_checkpointing:
            self.base.gradient_checkpointing_enable()
            self.base.config.use_cache = False

        n_layers = len(self.base.h)
        n_frozen = min(max(cfg.freeze_bottom_n_layers, 0), n_layers)
        for block in self.base.h[:n_frozen]:
            for parameter in block.parameters():
                parameter.requires_grad_(False)

    def _build_inputs(self, text_ids, text_lens, audio_ids, audio_lens):
        B = text_ids.size(0)
        device = text_ids.device
        H = self.base.config.n_embd
        L = int((text_lens + audio_lens + 2).max().item())
        inputs = torch.zeros(B, L, H, device=device)
        mask = torch.zeros(B, L, dtype=torch.long, device=device)
        labels = torch.full((B, L), -100, dtype=torch.long, device=device)

        for i in range(B):
            tl = int(text_lens[i].item())
            al = int(audio_lens[i].item())
            end = tl + al + 2
            inputs[i, :tl] = self.base.wte(text_ids[i, :tl])
            inputs[i, tl] = self.audio_emb.weight[self.cfg.bos_id]
            inputs[i, tl + 1:tl + 1 + al] = self.audio_emb(audio_ids[i, :al])
            inputs[i, tl + 1 + al] = self.audio_emb.weight[self.cfg.eos_id]
            mask[i, :end] = 1
            labels[i, tl + 1:tl + 1 + al] = audio_ids[i, :al]
            labels[i, tl + 1 + al] = self.cfg.eos_id

        return inputs, mask, labels

    def forward(self, text_ids, text_lens, audio_ids, audio_lens):
        inputs, mask, labels = self._build_inputs(text_ids, text_lens, audio_ids, audio_lens)
        h = self.base(inputs_embeds=inputs, attention_mask=mask, return_dict=True).last_hidden_state
        logits = self.audio_head(h)
        loss = F.cross_entropy(
            logits[:, :-1, :].contiguous().float().view(-1, logits.size(-1)),
            labels[:, 1:].contiguous().view(-1),
            ignore_index=-100,
        )
        return {"loss": loss, "logits": logits}

    @torch.no_grad()
    def generate_audio(self, text_ids, max_new_tokens=400, temperature=0.9, top_k=50):
        self.eval()
        device = text_ids.device
        text_ids = text_ids.view(-1).long().to(device)
        text_emb = self.base.wte(text_ids).unsqueeze(0)
        bos = self.audio_emb.weight[self.cfg.bos_id].view(1, 1, -1)
        prefix = torch.cat([text_emb, bos], dim=1)
        mask = torch.ones(1, prefix.size(1), device=device, dtype=torch.long)
        out = self.base(inputs_embeds=prefix, attention_mask=mask, use_cache=True, return_dict=True)
        past = out.past_key_values
        h = out.last_hidden_state[:, -1, :]
        generated = []

        for _ in range(max_new_tokens):
            logits = self.audio_head(h) / max(temperature, 1e-6)
            if top_k is not None and top_k > 0 and top_k < logits.size(-1):
                kth = torch.topk(logits, top_k, dim=-1).values[:, -1, None]
                logits = logits.masked_fill(logits < kth, float("-inf"))
            next_token = torch.multinomial(torch.softmax(logits, dim=-1), num_samples=1).squeeze(-1)
            token_id = int(next_token.item())
            if token_id == self.cfg.eos_id:
                break
            generated.append(token_id)
            token_emb = self.audio_emb(next_token).view(1, 1, -1)
            mask = torch.ones(1, prefix.size(1) + len(generated), device=device, dtype=torch.long)
            out = self.base(
                inputs_embeds=token_emb,
                attention_mask=mask,
                past_key_values=past,
                use_cache=True,
                return_dict=True,
            )
            past = out.past_key_values
            h = out.last_hidden_state[:, -1, :]

        return torch.tensor(generated, dtype=torch.long, device=device)



In [12]:
tts_cfg = GPT2TTSConfig(
    base_model=cfg.base_model,
    codebook_size=CODEBOOK_SIZE,
    freeze_bottom_n_layers=cfg.freeze_bottom_n_layers,
    tie_audio_weights=cfg.tie_audio_weights,
    gradient_checkpointing=cfg.gradient_checkpointing,
)
tts = GPT2TTS(tts_cfg)

trainable_params = sum(p.numel() for p in tts.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in tts.parameters())
print(f"trainable params: {trainable_params:,} / {total_params:,}")

test_batch = {k: v for k, v in batch.items() if torch.is_tensor(v)}
with torch.no_grad():
    out = tts(**test_batch)

print(f"Loss: {out['loss'].item()}, Logits Shape: {tuple(out['logits'].shape)}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

trainable params: 116,557,056 / 130,732,800
Loss: 12.356011390686035, Logits Shape: (8, 279, 8194)


## Training

In [13]:
class LitGPT2TTS(pl.LightningModule):
    def __init__(self, tts_cfg: GPT2TTSConfig, learning_rate=2e-5, weight_decay=0.01, warmup_fraction=0.1):
        super().__init__()
        self.save_hyperparameters()
        self.model = GPT2TTS(tts_cfg)
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.warmup_fraction = warmup_fraction

    def forward(self, **batch):
        return self.model(
            text_ids=batch["text_ids"],
            text_lens=batch["text_lens"],
            audio_ids=batch["audio_ids"],
            audio_lens=batch["audio_lens"],
        )

    def training_step(self, batch, batch_idx):
        out = self(**batch)
        self.log("train_loss", out["loss"], prog_bar=True, on_step=True, on_epoch=True, batch_size=batch["text_ids"].size(0))
        return out["loss"]

    def validation_step(self, batch, batch_idx):
        out = self(**batch)
        self.log("val_loss", out["loss"], prog_bar=True, on_step=False, on_epoch=True, batch_size=batch["text_ids"].size(0))
        return out["loss"]

    def configure_optimizers(self):
        no_decay = ["bias", "ln_", "LayerNorm.weight"]
        decay_params = []
        no_decay_params = []

        for name, parameter in self.named_parameters():
            if not parameter.requires_grad:
                continue
            if any(pattern in name for pattern in no_decay):
                no_decay_params.append(parameter)
            else:
                decay_params.append(parameter)

        optimizer = torch.optim.AdamW(
            [
                {"params": decay_params, "weight_decay": self.weight_decay},
                {"params": no_decay_params, "weight_decay": 0.0},
            ],
            lr=self.learning_rate,
            betas=(0.9, 0.95),
        )

        total_steps = max(int(self.trainer.estimated_stepping_batches), 1)
        warmup_fraction = min(max(float(self.warmup_fraction), 0.0), 0.5)
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=self.learning_rate,
            total_steps=total_steps,
            pct_start=warmup_fraction,
            anneal_strategy="cos",
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {"scheduler": scheduler, "interval": "step"},
        }



In [14]:
lit_model = LitGPT2TTS(
    tts_cfg=tts_cfg,
    learning_rate=cfg.learning_rate,
    weight_decay=cfg.weight_decay,
    warmup_fraction=cfg.warmup_fraction,
)

checkpoint_callback = ModelCheckpoint(
    dirpath=cfg.checkpoint_dir,
    filename="gpt2tts-{epoch:02d}-{val_loss:.3f}",
    monitor="val_loss",
    mode="min",
    save_top_k=2,
    save_last=True,
)

trainer = pl.Trainer(
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    max_epochs=cfg.max_epochs,
    precision=cfg.precision,
    accumulate_grad_batches=cfg.accumulate_grad_batches,
    gradient_clip_val=cfg.gradient_clip_val,
    callbacks=[checkpoint_callback],
    default_root_dir=str(cfg.checkpoint_dir),
    log_every_n_steps=10,
    logger=False,
)



Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [15]:
trainer.fit(lit_model, train_dl, val_dl)
print(checkpoint_callback.best_model_path)

/home/gllekkpc/env/lib/python3.13/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /home/gllekkpc/Documents/AudioML_HW_pavlosiuk_denysova/artifacts/checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.
/home/gllekkpc/env/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/gllekkpc/env/lib/python3.13/site-packages/pytorch_lightning/utilities/model_summary/model_summary.py:242: Precision bf16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ GPT2TTS │  130 M │ train │     0 │
└───┴───────┴─────────┴────────┴───────┴───────┘

Trainable params: 116 M                                                                                            
Non-trainable params: 14.2 M                                                                                       
Total params: 130 M                                                                                                
Total estimated model params size (MB): 522                                                                        
Modules in train mode: 3                                                                                           
Modules in eval mode: 162                                                                                          
Total FLOPs: 0

Output()

/home/gllekkpc/env/lib/python3.13/site-packages/pytorch_lightning/loops/fit_loop.py:534: Found 162 module(s) in 
eval mode at the start of training. This may lead to unexpected behavior during training. If this is intentional, 
you can ignore this warning.

`Trainer.fit` stopped: `max_epochs=10` reached.


/home/gllekkpc/Documents/AudioML_HW_pavlosiuk_denysova/artifacts/checkpoints/gpt2tts-epoch=09-val_loss=7.852.ckpt


In [16]:
experiment_cfg = asdict(cfg)
for key in ["dataset_dir", "cache_dir", "checkpoint_dir"]:
    experiment_cfg[key] = str(experiment_cfg[key])

torch.save(
    {
        "model_state_dict": lit_model.model.state_dict(),
        "model_cfg": asdict(tts_cfg),
        "experiment_cfg": experiment_cfg,
        "tokenizer_name": cfg.base_model,
        "base_model": cfg.base_model,
        "codebook_size": CODEBOOK_SIZE,
        "codec_name": cfg.codec_name,
        "codec_sample_rate": CODEC_SR,
    },
    cfg.checkpoint_dir / "gpt2tts_last_state_dict.pt",
)